# Модель для прогнозирования оттока клиентов для сервиса доставки кофе

---

## Контекст и цель проекта
Happy Beans Coffee запустила сервис доставки кофе и столкнулась с регулярным оттоком клиентов. В условиях высокой конкуренции это критично для бизнеса: удержание существующих клиентов обходится существенно дешевле, чем привлечение новых.
Цель проекта — построить интерпретируемую модель бинарной классификации, которая прогнозирует вероятность оттока клиента в следующем месяце.

## Бизнес-задача
Для каждого текущего клиента определить один из двух статусов:
- `0` — клиент остаётся;
- `1` — клиент может уйти.

Прогноз будет использоваться для:
- точечных retention-акций (скидки, промокоды, персональные предложения);
- более эффективного распределения маркетингового бюджета;
- повышения точности планирования выручки и оценки клиентской базы.

## Данные
В проекте используется датасет `coffee_churn_dataset.csv`, где каждая строка — один клиент, а признаки агрегированы за последние 4 недели на основе внутренних систем компании (транзакции, логи приложения, опросы).
Целевая переменная: `churn`.

## Описание признаков датасета

| Признак | Описание |
|---|---|
| `user_id` | Идентификатор пользователя |
| `days_since_last_order` | Количество дней с последнего заказа |
| `order_frequency_month` | Среднее число заказов в месяц |
| `order_frequency_week` | Среднее число заказов в неделю |
| `avg_order_value` | Средний чек, руб. |
| `median_order_value` | Медианный чек, руб. |
| `total_spent_last_month` | Сумма заказов за последний месяц, руб. |
| `total_spent_last_week` | Сумма заказов за последнюю неделю, руб. |
| `discount_usage_rate` | Доля заказов со скидкой за последний месяц |
| `last_coffee_type` | Сорт кофе в последнем заказе |
| `preferred_roast` | Предпочитаемый тип обжарки |
| `milk_preference` | Предпочитаемый тип молока |
| `seasonal_menu_tried` | Пробовал ли клиент сезонное меню (`0/1`) |
| `coffee_bean_origin` | Страна происхождения зерна |
| `last_drink_size` | Размер напитка в последнем заказе |
| `subscription_status` | Тип подписки клиента |
| `app_opens_per_week` | Среднее число открытий приложения в неделю |
| `notifications_enabled` | Включены ли push-уведомления (`0/1`) |
| `review_rating_last_10` | Средняя оценка последних 10 заказов |
| `review_rating_last_1` | Оценка последнего заказа |
| `app_crashes_last_month` | Количество зависаний приложения за месяц |
| `seasons` | Текущее время года |
| `days_since_last_promo` | Дни с момента последнего использования промо/акции |
| `phone_type` | Тип устройства клиента |
| `coffee_preference_change` | Менялись ли вкусовые предпочтения (`0/1`) |
| `geo_id` | Идентификатор региона |
| `churn` | Целевая переменная: факт оттока (`1` — ушёл, `0` — остался) |

`user_id` — технический идентификатор, в обучении модели не используется.

## Подход к моделированию
Решается задача бинарной классификации с акцентом на интерпретируемость модели.
Планируется сравнение baseline-модели (`DummyClassifier`) и линейной модели (`LogisticRegression`) с кросс-валидацией, предобработкой пропусков и последующим подбором гиперпараметров.

## Метрики качества
Основная метрика: `PR AUC`, так как задача несбалансированная и важно качество предсказаний класса оттока.
Дополнительные метрики: `Precision`, `Recall`, `F1` для анализа компромисса между охватом и точностью retention-кампаний.

## План работы
1. Подготовка среды и библиотек.
2. EDA и первичный анализ качества данных.
3. Предобработка данных и построение пайплайна.
4. Обучение baseline-модели.
5. Генерация и отбор признаков.
6. Подбор гиперпараметров `LogisticRegression`.
7. Финальная оценка на тесте и сохранение модели с пайплайном для внедрения.

## Этап 1. Подготовка среды и библиотек


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from phik import phik_matrix

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, classification_report


RANDOM_STATE = 42
DATA_PATH = Path('coffee_churn_dataset.csv')
FALLBACK_DATA_PATH = Path('/datasets/coffee_churn_dataset.csv')

if not DATA_PATH.exists() and FALLBACK_DATA_PATH.exists():
    DATA_PATH = FALLBACK_DATA_PATH

df = pd.read_csv(DATA_PATH, sep=',', decimal='.')

---

## Этап 2. Первичный анализ данных

### Описание данных

In [ ]:
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок")
display(df.head(5))
display(df.sample(5, random_state=RANDOM_STATE))
display(df.dtypes.to_frame("dtype"))

- В датасете 10450 строк и 27 признаков.
- Данные содержат числовые и категориальные признаки, каждая строка соответствует одному клиенту.
- Есть поведенческие признаки (частота заказов, траты, активность в приложении), продуктовые предпочтения и сервисные признаки.

### Анализ целевой переменной `churn`

In [ ]:
target_count = df['churn'].value_counts().sort_index()
target_share = (df['churn'].value_counts(normalize=True).sort_index() * 100).round(2)

target_report = pd.concat(
    [target_count.rename('count'), target_share.rename('share_%')],
    axis=1
)
display(target_report)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='churn')
plt.title('Распределение целевой переменной churn')
plt.xlabel('Класс churn (0 — остался, 1 — ушел)')
plt.ylabel('Количество клиентов')
plt.show()

- Целевая переменная бинарная: 0 — клиент остался, 1 — клиент ушел.
- Наблюдается выраженный дисбаланс классов: доля оттока составляет только 6.02%.
- Из-за дисбаланса accuracy не подходит как основная метрика; в дальнейшем фокусируемся на Precision, Recall, F1 и особенно PR AUC.

### Признаки

In [ ]:
target_col = 'churn'

cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()

if target_col in num_cols:
    num_cols.remove(target_col)

print(f"Категориальные признаки ({len(cat_cols)}):")
print(cat_cols)
print()
print(f"Числовые признаки ({len(num_cols)}):")
print(num_cols)

# Проверим уникальность user_id
print()
print("Уникальных user_id:", df['user_id'].nunique())
print("Всего строк:", len(df))
print("user_id уникален:", df['user_id'].nunique() == len(df))

- Признаки представлены двумя типами: числовые и категориальные.
- `user_id` — технический идентификатор клиента, он уникален для каждой строки и не несет поведенческого сигнала для обобщения модели.
- На этапе моделирования `user_id` нужно исключить из набора признаков, чтобы избежать шума и переобучения.
- Остальные признаки содержательно связаны с поведением клиента и на текущем этапе сохраняются.

In [ ]:
# Разбор бинарных признаков
target_col = "churn"
binary_cols = ["seasonal_menu_tried", "notifications_enabled", "coffee_preference_change"]

print("Бинарные признаки:", binary_cols)
for col in binary_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())
    print("Доля churn по категориям:")
    print(df.groupby(col, dropna=False)[target_col].mean().rename("churn_rate"))


- Бинарные признаки `seasonal_menu_tried`, `notifications_enabled`, `coffee_preference_change` выделены в отдельную группу и проанализированы отдельно от непрерывных числовых.
- По всем трём признакам доля оттока в группах `0` и `1` близка (различия небольшие), то есть сильного самостоятельного эффекта на `churn` на уровне простого среза не наблюдается.
- Во всех бинарных признаках присутствуют пропуски (`NaN`), поэтому на этапе предобработки нужна импутация.
- Несмотря на слабые различия в EDA, признаки сохраняются для модели: их вклад может проявиться в комбинации с другими факторами.

In [ ]:
# Проверка отрицательных значений в числовых признаках
num_cols = df.select_dtypes(exclude="object").columns.tolist()
if target_col in num_cols:
    num_cols.remove(target_col)

neg_stats = (
    (df[num_cols] < 0)
    .sum()
    .rename("negative_count")
    .to_frame()
)
neg_stats["negative_share"] = (neg_stats["negative_count"] / len(df)).round(4)
neg_stats = neg_stats[neg_stats["negative_count"] > 0].sort_values("negative_count", ascending=False)

print("Признаки с отрицательными значениями:")
display(neg_stats)

- В ряде числовых признаков обнаружены отрицательные значения.
- Для признаков, где отрицательные значения физически невозможны (дни, суммы, частоты, число сбоев), такие наблюдения считаются аномалиями данных.
- На этапе предобработки отрицательные значения заменяются на `NaN`, после чего заполняются медианой в пайплайне.
- Это позволяет сохранить строки и одновременно снизить влияние некорректных значений на модель.

In [ ]:
# Логика обработки отрицательных значений (для этапа предобработки)
non_negative_cols = [
    "days_since_last_order", "order_frequency_month", "order_frequency_week",
    "avg_order_value", "median_order_value", "total_spent_last_month",
    "total_spent_last_week", "discount_usage_rate", "app_opens_per_week",
    "review_rating_last_10", "review_rating_last_1", "app_crashes_last_month",
    "days_since_last_promo"
]

X = df.copy()
X[non_negative_cols] = X[non_negative_cols].mask(X[non_negative_cols] < 0)

- Для признаков, где отрицательные значения физически невозможны, считаем их аномалиями -> NaN,
- затем заполняем медианой в пайплайне.

### Анализ пропусков

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_share_%': (missing / len(df) * 100).round(2)
})

display(missing_report)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=missing_report.reset_index(),
    x='index',
    y='missing_share_%',
    color='steelblue'
)
plt.xticks(rotation=90)
plt.title('Доля пропусков по признакам, %')
plt.xlabel('Признак')
plt.ylabel('Доля пропусков, %')
plt.tight_layout()
plt.show()

Пропуски присутствуют в большинстве признаков, но их доля умеренная: от 1.05% (geo_location) до 9.46% (seasonal_menu_tried). Критически высоких значений (например, 30%+) нет, поэтому удалять признаки или строки из-за пропусков нецелесообразно: это приведет к ненужной потере информации. Для дальнейшего моделирования оптимально использовать импутацию в пайплайне: для числовых признаков заполнение медианой, для категориальных заполнение наиболее частой категорией (или unknown). Импутацию нужно делать только после разделения на train/test, чтобы избежать утечки данных.

### Анализ категориальных признаков

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

cat_report = pd.DataFrame({
    'feature': cat_cols,
    'n_unique': [df[c].nunique(dropna=True) for c in cat_cols],
    'top_1_value': [df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else np.nan for c in cat_cols],
    'top_1_freq_%': [
        round(df[c].value_counts(normalize=True, dropna=True).iloc[0] * 100, 2)
        if not df[c].value_counts(normalize=True, dropna=True).empty else np.nan
        for c in cat_cols
    ]
}).sort_values('n_unique', ascending=False)

display(cat_report)

В датасете выделено 10 категориальных признаков.
Большинство из них имеют низкую или среднюю кардинальность (3–6 уникальных значений), поэтому их корректно кодировать с помощью `One-Hot Encoding`.

Признак `geo_location` имеет повышенную кардинальность (100 уникальных значений), что увеличит размерность после кодирования. На текущем этапе его можно сохранить и обрабатывать через `OneHotEncoder(handle_unknown='ignore')`, чтобы не терять потенциально важный региональный сигнал.

Признак `user_id` является техническим идентификатором и уникален для каждой строки (`10450` из `10450`), поэтому для моделирования он не подходит и должен быть исключён.

На этапе экспериментов с признаками целесообразно отдельно сравнить качество модели для двух вариантов `geo_location`: базовый `OHE` и альтернативный `frequency encoding`.

### Анализ выбросов

Для числовых признаков необходимо проверить наличие выбросов, так как они могут искажать масштабирование и влиять на коэффициенты линейной модели.

Для этого используем визуализации (boxplot) и статистический подход по межквартильному размаху (IQR).
На текущем этапе выбросы не удаляем из датасета вручную, чтобы не потерять полезные наблюдения. Вместо этого в дальнейшем обработаем их внутри пайплайна (например, через `RobustScaler` или винзоризацию по квантилям).

Такой подход позволяет:
- сохранить максимум данных;
- снизить влияние экстремальных значений;
- избежать утечки данных, если обработка выполняется только на train-

In [ ]:
# Числовые признаки для анализа выбросов (без таргета и бинарных)
target_col = "churn"
binary_cols = ["seasonal_menu_tried", "notifications_enabled", "coffee_preference_change"]

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in [target_col] + binary_cols]

# IQR-отчет
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_mask = (df[numeric_cols] < lower) | (df[numeric_cols] > upper)
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
outlier_share = (outlier_counts / len(df) * 100).round(2)

outlier_report = (
    pd.DataFrame(
        {
            "outlier_count": outlier_counts,
            "outlier_share_%": outlier_share,
        }
    )
    .query("outlier_count > 0")
)

display(outlier_report.head(15))

# Boxplot для топ-6 признаков с наибольшей долей выбросов
top_n = min(6, len(outlier_report))
top_outlier_cols = outlier_report.head(top_n).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, col in enumerate(top_outlier_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="skyblue")
    axes[i].set_title(col)

for j in range(top_n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

По IQR-проверке выбросы присутствуют в ряде непрерывных числовых признаков, особенно в:
- `total_spent_last_month`
- `total_spent_last_week`
- `days_since_last_order`
- `days_since_last_promo`
- `app_opens_per_week`
- `avg_order_value`

Важно: бинарные признаки `seasonal_menu_tried`, `notifications_enabled`, `coffee_preference_change` анализируются отдельно и не включаются в IQR-проверку, так как для них такой критерий неинформативен.

Для признаков с длинными хвостами:
- не удаляем строки вручную, чтобы не терять данные;
- обрабатываем признаки внутри пайплайна после train/test split;
- используем `RobustScaler`, так как он масштабирует по медиане и межквартильному размаху (IQR), а значит значительно менее чувствителен к выбросам, чем `StandardScaler`, который опирается на среднее и стандартное отклонение;
- отрицательные значения в признаках, где они физически невозможны, обрабатываем отдельно по схеме: `< 0 -> NaN -> median`.

Итог: выбросы и аномалии контролируемы, их обработка встроена в воспроизводимый пайплайн без утечки данных.

### Корреляция

In [ ]:
# 1) Связь признаков между собой
df_phik = df.drop(columns=["user_id"]).copy()

phik_corr = df_phik.phik_matrix(
    interval_cols=[c for c in df_phik.columns if df_phik[c].dtype != "object" and c != "churn"]
)

plt.figure(figsize=(14, 10))
sns.heatmap(phik_corr, cmap="coolwarm", vmin=0, vmax=1)
plt.title("PhiK: связь признаков между собой")
plt.show()

In [ ]:
# 2) Связь признаков с целевой переменной churn
phik_to_target = (
    phik_corr["churn"]
    .drop("churn")
    .sort_values(ascending=False)
    .to_frame("phik_with_churn")
)

display(phik_to_target.head(15))

По матрице `PhiK` выраженной мультиколлинеарности между большинством признаков не наблюдается: в основном связи слабые. Более заметные зависимости есть только у признаков, которые близки по смыслу (частоты заказов и траты), что ожидаемо.

Связь признаков с целевой переменной `churn` (по `PhiK`) в целом слабая, кроме `app_crashes_last_month`:
- `app_crashes_last_month`: `0.854` — очень сильная связь;
- `app_opens_per_week`: `0.126`, `subscription_status`: `0.105`, `order_frequency_week/month`: около `0.08` — слабая/умеренная связь;
- у большинства остальных признаков связь с `churn` слабая или близка к нулю.

Отдельно проверим `app_crashes_last_month` на этапе моделирования и интерпретации, так как столь высокая связь может означать как сильный бизнес-сигнал, так и потенциальный риск утечки/артефакта данных. На текущем этапе признаки не удаляем только по корреляции: финальный отбор делаем по качеству модели на кросс-валидации.

## Выводы по результатам EDA

Датасет содержит 10450 наблюдений и 27 колонок. Целевая переменная `churn` бинарная и несбалансирована: класс оттока составляет около 6.02%, поэтому при обучении модели основной фокус нужно делать на метриках `Precision`, `Recall`, `F1` и `PR AUC`.

В данных присутствуют пропуски (примерно от 1% до 9.5% в разных признаках), но критически высоких долей нет, поэтому удаление строк/признаков не требуется. Для дальнейшей работы выбрана стратегия импутации в пайплайне: числовые признаки заполнять медианой, категориальные — наиболее частым значением (или `unknown`).

Категориальные признаки в основном имеют низкую/среднюю кардинальность и подходят для `One-Hot Encoding`. Признак `geo_location` (100 категорий) сохраняем и кодируем через `OneHotEncoder(handle_unknown='ignore')`; альтернативное кодирование (`frequency encoding`) проверим на этапе экспериментов. Признак `user_id` исключаем как технический идентификатор.

В числовых признаках обнаружены выбросы. Для устойчивой обработки в линейной модели используем масштабирование, менее чувствительное к выбросам (`RobustScaler`) внутри пайплайна. По `PhiK` самая выраженная связь с оттоком наблюдается у `app_crashes_last_month`, остальные признаки в основном имеют слабую или умеренную связь с `churn`.

---

## Этап 3. Предобработка данных


In [ ]:
TARGET = "churn"
ID_COL = "user_id"

X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train mean:", round(y_train.mean(), 4))
print("y_test mean:", round(y_test.mean(), 4))

In [ ]:
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

geo_col = "geo_location"
cat_wo_geo = [c for c in cat_cols if c != geo_col]

# Колонки, где отрицательные значения считаем аномалиями
non_negative_cols = [
    "days_since_last_order", "order_frequency_month", "order_frequency_week",
    "avg_order_value", "median_order_value", "total_spent_last_month",
    "total_spent_last_week", "discount_usage_rate", "app_opens_per_week",
    "review_rating_last_10", "review_rating_last_1", "app_crashes_last_month",
    "days_since_last_promo"
]
non_negative_cols = [c for c in non_negative_cols if c in num_cols]

def replace_negative_with_nan(X):
    X = pd.DataFrame(X).copy()
    X = X.mask(X < 0)
    return X

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        s = pd.Series(X.iloc[:, 0] if hasattr(X, "iloc") else X[:, 0])
        self.freq_map_ = s.value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        s = pd.Series(X.iloc[:, 0] if hasattr(X, "iloc") else X[:, 0])
        return s.map(self.freq_map_).fillna(0.0).to_frame("geo_location_freq")

    def get_feature_names_out(self, input_features=None):
        return np.array(["geo_location_freq"], dtype=object)

numeric_nonneg_pipeline = Pipeline(steps=[
    ("neg_to_nan", FunctionTransformer(replace_negative_with_nan, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

numeric_other_cols = [c for c in num_cols if c not in non_negative_cols]
numeric_other_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

geo_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("freq", FrequencyEncoder())
])

transformers = [
    ("num_nonneg", numeric_nonneg_pipeline, non_negative_cols),
    ("cat", categorical_pipeline, cat_wo_geo),
    ("geo", geo_pipeline, [geo_col]),
]
if numeric_other_cols:
    transformers.append(("num_other", numeric_other_pipeline, numeric_other_cols))

preprocessor = ColumnTransformer(transformers=transformers)

print("Числовые (non-negative):", len(non_negative_cols))
print("Числовые (прочие):", len(numeric_other_cols))
print("Категориальные (OHE):", len(cat_wo_geo))
print("geo_location: frequency encoding")

---

## Этап 4. Обучение модели


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "pr_auc": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

# Baseline 1: Dummy
dummy_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE))
])

dummy_cv = cross_validate(
    dummy_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Baseline 2: LogisticRegression
logreg_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=2000,
        class_weight="balanced"
    ))
])

logreg_cv = cross_validate(
    logreg_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

def cv_summary(cv_result, model_name):
    return {
        "model": model_name,
        "PR_AUC_mean": cv_result["test_pr_auc"].mean(),
        "F1_mean": cv_result["test_f1"].mean(),
        "Precision_mean": cv_result["test_precision"].mean(),
        "Recall_mean": cv_result["test_recall"].mean(),
    }

report = pd.DataFrame([
    cv_summary(dummy_cv, "DummyClassifier"),
    cv_summary(logreg_cv, "LogisticRegression")
])

display(report)

### Вывод по обучению базовой модели

`DummyClassifier` показал ожидаемо низкое качество: `PR AUC = 0.060`, `F1 = 0.000`, `Precision = 0.000`, `Recall = 0.000`. Модель фактически не выявляет класс оттока.

`LogisticRegression` значительно превосходит baseline:
- `PR AUC = 0.631`
- `F1 = 0.427`
- `Precision = 0.285`
- `Recall = 0.855`

Модель хорошо покрывает класс оттока (высокий `Recall`), но `Precision` умеренная, то есть доля ложноположительных срабатываний остаётся заметной.

Итог: базовой рабочей моделью выбираем `LogisticRegression`. Далее улучшаем качество через feature engineering и подбор гиперпараметров, ориентируясь в первую очередь на `PR AUC`.

---

## Этап 5. Создание новых признаков


### Генерация новых признаков

In [ ]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    X_new = data.copy()

    eps = 1e-6  # защита от деления на 0

    # Интенсивность и "ценность" заказов
    X_new["spent_per_order_month"] = X_new["total_spent_last_month"] / (X_new["order_frequency_month"] + eps)
    X_new["spent_per_order_week"] = X_new["total_spent_last_week"] / (X_new["order_frequency_week"] + eps)

    # Трансформации
    X_new["days_since_last_order_sqrt"] = np.sqrt(np.clip(X_new["days_since_last_order"], a_min=0, a_max=None))
    X_new["app_crashes_last_month_sq"] = X_new["app_crashes_last_month"] ** 2
    X_new["days_since_last_promo_sq"] = X_new["days_since_last_promo"] ** 2

    # Поведенческий индикатор: "давно не заказывал и мало открывает приложение"
    X_new["inactivity_ratio"] = X_new["days_since_last_order"] / (X_new["app_opens_per_week"] + 1)

    return X_new

X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)

print("До FE:", X_train.shape)
print("После FE:", X_train_fe.shape)

### Обновляем пайплайн и сравниваем CV (без FE vs c FE)

In [ ]:
def build_preprocessor(cat_cols, num_cols, geo_col="geo_location"):
    cat_wo_geo = [c for c in cat_cols if c != geo_col]

    nonneg = [c for c in non_negative_cols if c in num_cols]
    num_other = [c for c in num_cols if c not in nonneg]

    transformers = [
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_wo_geo),
        ("geo", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("freq", FrequencyEncoder())
        ]), [geo_col]),
    ]

    if nonneg:
        transformers.append((
            "num_nonneg",
            Pipeline([
                ("neg_to_nan", FunctionTransformer(replace_negative_with_nan, validate=False, feature_names_out="one-to-one")),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            nonneg
        ))

    if num_other:
        transformers.append((
            "num_other",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            num_other
        ))

    return ColumnTransformer(transformers=transformers)

cat_cols_base = X_train.select_dtypes(include="object").columns.tolist()
num_cols_base = X_train.select_dtypes(exclude="object").columns.tolist()

cat_cols_fe = X_train_fe.select_dtypes(include="object").columns.tolist()
num_cols_fe = X_train_fe.select_dtypes(exclude="object").columns.tolist()

preprocessor_base = build_preprocessor(cat_cols_base, num_cols_base)
preprocessor_fe = build_preprocessor(cat_cols_fe, num_cols_fe)

model_params = dict(random_state=RANDOM_STATE, max_iter=2000, class_weight="balanced")

pipe_base = Pipeline([
    ("preprocessor", preprocessor_base),
    ("model", LogisticRegression(**model_params))
])

pipe_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", LogisticRegression(**model_params))
])

base_cv = cross_validate(pipe_base, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
fe_cv = cross_validate(pipe_fe, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=-1)

compare = pd.DataFrame([
    cv_summary(base_cv, "LogReg_base"),
    cv_summary(fe_cv, "LogReg_with_FE"),
])
display(compare)

### Интерпретация коэффициентов для модели с FE

In [ ]:
pipe_fe.fit(X_train_fe, y_train)

feature_names = pipe_fe.named_steps["preprocessor"].get_feature_names_out()
coefs = pipe_fe.named_steps["model"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})
coef_df["abs_coef"] = coef_df["coef"].abs()

top_positive = coef_df.sort_values("coef", ascending=False).head(15)
top_negative = coef_df.sort_values("coef", ascending=True).head(15)

print("Топ признаков, повышающих вероятность churn:")
display(top_positive[["feature", "coef"]])

print("Топ признаков, снижающих вероятность churn:")
display(top_negative[["feature", "coef"]])

### Вывод по этапу 5 (Feature Engineering)

На этапе feature engineering были добавлены производные поведенческие и транзакционные признаки (`spent_per_order_*`, `sqrt`/`square`-преобразования, `inactivity_ratio`), после чего модель повторно оценена на кросс-валидации.

Модель с новыми признаками (`LogReg_with_FE`) улучшила качество относительно базовой версии:
- `PR AUC`: `0.631` → `0.671` (`+0.040`)
- `F1`: `0.427` → `0.473`
- `Precision`: `0.285` → `0.329`
- `Recall`: `0.855` → `0.841` (незначительное снижение)

Так как целевая метрика проекта — `PR AUC`, рабочей конфигурацией выбираем `LogReg_with_FE` и используем её на этапе подбора гиперпараметров.

По коэффициентам модели наибольший вклад в рост вероятности оттока дают признаки, связанные с нестабильностью приложения (`app_crashes_last_month`, `app_crashes_last_month_sq`) и статусом подписки; признаки со знаком «минус» ассоциированы со снижением риска оттока.

---

## Этап 6. Эксперименты с гиперпараметрами


### Гиперпараметры для экспериментов (`LogisticRegression`)

В рамках систематического подбора планируется проверить следующие гиперпараметры:

- `C` — коэффициент обратной регуляризации (чем меньше значение, тем сильнее регуляризация).
- `penalty` — тип регуляризации (`l1`, `l2`).
- `solver` — алгоритм оптимизации (`liblinear`, `lbfgs`).
- `class_weight` — учёт дисбаланса классов (`balanced` и, при необходимости, `None`).
- `max_iter` — максимальное число итераций оптимизатора (для устойчивой сходимости).

Основной метрикой отбора является `PR AUC` (`average_precision`), так как задача несбалансированная и важен корректный поиск класса оттока.

### Подход к подбору гиперпараметров

Для `LogisticRegression` будет выполнен систематический перебор гиперпараметров с помощью `GridSearchCV`.
Оценка каждой конфигурации проводится на стратифицированной кросс-валидации (`StratifiedKFold`, 5 фолдов), чтобы сохранить долю класса оттока в каждом фолде и получить стабильную оценку качества.

В качестве целевой метрики используется `PR AUC` (`average_precision`), так как в задаче наблюдается выраженный дисбаланс классов (`churn=1` — миноритарный класс), и важно корректно оценивать качество именно по клиентам с риском оттока

In [ ]:
pipe_tune = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=3000,
        class_weight="balanced"
    ))
])

param_grid = [
    {
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.01, 0.1, 1, 5, 10],
    },
    {
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.1, 1, 5, 10],
    }
]

grid = GridSearchCV(
    estimator=pipe_tune,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

grid.fit(X_train_fe, y_train)

print("Best params:", grid.best_params_)
print("Best CV PR AUC:", round(grid.best_score_, 6))

In [ ]:
results_table = pd.DataFrame(grid.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")

results_table = results_table.rename(columns={
    "mean_test_score": "mean_pr_auc",
    "std_test_score": "std_pr_auc"
})

display(results_table.head(15))

### Результаты подбора гиперпараметров

Для `LogisticRegression` выполнен перебор 15 конфигураций (`75` запусков: 5 фолдов × 15 кандидатов) с метрикой отбора `PR AUC`.

Лучшая конфигурация:
- `solver = liblinear`
- `penalty = l1`
- `C = 0.1`

Лучшее качество на кросс-валидации:
- `PR AUC = 0.6805` (std = `0.0443`)

Результаты показывают, что после подбора гиперпараметров модель улучшилась относительно конфигурации до тюнинга (`PR AUC = 0.6714`) и показывает стабильное качество на кросс-валидации.
Эта конфигурация используется для финального обучения и оценки на тестовой выборке.

---

## Этап 7. Подготовка финальной модели


In [ ]:
# Лучшая модель из этапа 6
final_model = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        penalty="l1",
        C=0.1
    ))
])

# Обучение на train
final_model.fit(X_train_fe, y_train)

# Предсказания на test
y_test_proba = final_model.predict_proba(X_test_fe)[:, 1]
y_test_pred = final_model.predict(X_test_fe)

# Метрики на test
test_metrics = {
    "PR_AUC": average_precision_score(y_test, y_test_proba),
    "F1": f1_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
}

pd.DataFrame([test_metrics]).round(6)

In [ ]:
print(classification_report(y_test, y_test_pred, digits=4))

### Этап 7. Подготовка финальной модели

Лучшая конфигурация (`LogisticRegression`, `solver='liblinear'`, `penalty='l1'`, `C=0.1`) обучена на тренировочной выборке с feature engineering и предобработкой (`geo_location` через frequency encoding, остальные категориальные признаки через OHE).

Качество на тестовой выборке:
- `PR AUC = 0.7330`
- `F1 = 0.5176`
- `Precision = 0.3679`
- `Recall = 0.8730`

Модель показывает высокий `Recall` для класса оттока (находит большую часть потенциально уходящих клиентов) при умеренном `Precision`, что соответствует задаче раннего выявления клиентов группы риска.
По ключевой метрике `PR AUC` качество выше, чем на кросс-валидации, что говорит о хорошей обобщающей способности выбранной конфигурации.

---

## Этап 8. Отчёт о проделанной работе

В проекте построена модель прогнозирования оттока клиентов сервиса доставки кофе.
Задача решалась как бинарная классификация с фокусом на метрику `PR AUC`, так как класс оттока несбалансирован (около 6%).

Ключевые шаги, повлиявшие на качество модели:
- обработка пропусков в воспроизводимом пайплайне;
- явная обработка отрицательных значений в признаках, где они физически невозможны (`< 0 -> NaN -> median`);
- генерация новых поведенческих и транзакционных признаков;
- устойчивое масштабирование числовых признаков (`RobustScaler`);
- кодирование `geo_location` через frequency encoding;
- подбор гиперпараметров `LogisticRegression` по кросс-валидации.

Финальная модель на тестовой выборке:
- `PR AUC = 0.7330`
- `F1 = 0.5176`
- `Precision = 0.3679`
- `Recall = 0.8730`

Модель хорошо выявляет клиентов с риском оттока (высокий `Recall`), что полезно для retention-кампаний.
При этом `Precision` остаётся умеренным, поэтому для практического внедрения рекомендуется дополнительно настроить порог классификации под допустимый бюджет удержания.

---

## Этап 9. Сохранение модели для продакшена

Сохраните итоговую модель и пайплайн предобработки. Убедитесь, что всё работает: загрузите артефакты и проверьте их на тестовых данных. В решении укажите ссылку для скачивания сохранённых файлов.

In [ ]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

model_path = artifacts_dir / "churn_model_pipeline.joblib"

# сохраняем финальный pipeline целиком (предобработка + модель)
joblib.dump(final_model, model_path)

print("Saved:", model_path.resolve())

In [ ]:
loaded_model = joblib.load(model_path)

loaded_proba = loaded_model.predict_proba(X_test_fe)[:, 1]
loaded_pred = loaded_model.predict(X_test_fe)

check_metrics = {
    "PR_AUC": average_precision_score(y_test, loaded_proba),
    "F1": f1_score(y_test, loaded_pred),
    "Precision": precision_score(y_test, loaded_pred),
    "Recall": recall_score(y_test, loaded_pred),
}

pd.DataFrame([check_metrics]).round(6)

### Сохранённые артефакты

- Итоговый pipeline (предобработка + модель):
  `artifacts/churn_model_pipeline.joblib`
- Полный путь в текущем окружении:
  `D:\Dev\coffee-churn-prediction\artifacts\churn_model_pipeline.joblib`

### Проверка артефакта

После загрузки `joblib.load(...)` модель успешно делает предсказания на тестовых данных, метрики совпадают с финальной оценкой:
- `PR AUC = 0.733009`
- `F1 = 0.517647`
- `Precision = 0.367893`
- `Recall = 0.873016`